## streamflow eda
des plaines river at riverside (usgs 05532500): daily mean discharge + open-meteo weather. no modeling yet - just the data and its drift/quality hooks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sienna, olive, steel, gray = "#a0522d", "#808000", "#4682b4", "#b0b0b0"
plt.rcParams["axes.facecolor"] = "#faf7f2"
plt.rcParams["figure.facecolor"] = "#faf7f2"

df = pd.read_parquet("../data/raw/streamflow.parquet")
df["date"] = pd.to_datetime(df["date"])
df.shape

### overview
~82 years daily, no missing discharge or weather.

In [ ]:
print("span:", df.date.min().date(), "->", df.date.max().date())
print("rows:", len(df))
print("discharge nulls:", df.discharge_cfs.isna().sum(), "| weather nulls:", df.precip_mm.isna().sum())
df.head(3)

### target
0 to 10,700 cfs, heavily right-skewed -> model in log space. median ~365, floods 20-30x that.

In [ ]:
print(df.discharge_cfs.describe()[["min","50%","mean","max"]])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(df.discharge_cfs, bins=60, color=sienna)
ax[0].set_title("discharge (cfs)", weight="bold"); ax[0].set_ylabel("days")
ax[1].hist(np.log1p(df.discharge_cfs), bins=60, color=olive)
ax[1].set_title("log(1 + discharge)", weight="bold")
plt.tight_layout(); plt.show()

### hydrograph

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
ax[0].plot(df.date, df.discharge_cfs, color=steel, lw=0.4)
ax[0].set_yscale("log"); ax[0].set_title("full record (log cfs)", weight="bold")
recent = df[df.date >= df.date.max() - pd.Timedelta(days=730)]
ax[1].plot(recent.date, recent.discharge_cfs, color=sienna, lw=0.8)
ax[1].set_title("last 2 years (cfs)", weight="bold")
plt.tight_layout(); plt.show()

### rain -> flow
the learnable signal. july 2026 flood: 17-21mm rain, discharge 455 -> 3420 cfs.

In [ ]:
d = df.copy()
d["next_discharge"] = d.discharge_cfs.shift(-1)   # next-day target
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(d.precip_mm, np.log1p(d.next_discharge), s=3, alpha=0.15, color=olive)
ax[0].set_xlabel("precip today (mm)"); ax[0].set_ylabel("log next-day discharge")
ax[0].set_title("precip vs next-day flow", weight="bold")
flood = d[(d.date >= "2026-06-28") & (d.date <= "2026-07-10")]
ax[1].plot(flood.date, flood.discharge_cfs, color=sienna, marker="o", label="discharge")
ax[1].bar(flood.date, flood.precip_mm * 100, color=steel, alpha=0.4, label="precip (x100)")
ax[1].set_title("july 2026 flood", weight="bold"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### seasonality
spring high, late-summer low - natural concept drift for monitoring.

In [ ]:
d["month"] = d.date.dt.month
monthly = [d[d.month == m].discharge_cfs.values for m in range(1, 13)]
fig, ax = plt.subplots(figsize=(11, 3.8))
bp = ax.boxplot(monthly, showfliers=False, patch_artist=True)
for box in bp["boxes"]: box.set(facecolor=olive, alpha=0.6)
ax.set_xticklabels(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"])
ax.set_ylabel("discharge (cfs)"); ax.set_title("discharge by month", weight="bold")
plt.tight_layout(); plt.show()

### quality + drift hooks
all real, no simulation: `provisional` tail (usgs revises for months), `estimated`/`ice` winter qualifiers, `revised` = values usgs already changed.

In [ ]:
print("approval:", df.approval_status.value_counts().to_dict())
print("qualifier:", df.qualifier.value_counts(dropna=False).to_dict())
est = df[df.qualifier.notna() & df.qualifier.str.contains("ESTIMATED|ICE")]
print("estimated/ice by month:", est.date.dt.month.value_counts().sort_index().to_dict())
prov = df[df.approval_status == "Provisional"]
print("provisional tail:", prov.date.min().date(), "->", prov.date.max().date(), f"({len(prov)} rows)")

### takeaways
- log space, persistence baseline, nse + log-rmse
- features: discharge lags/rolling, precip + temp, day-of-year
- monitoring: seasonal regimes, floods, winter estimates, provisional revisions
- next: `features.py`, then train + mlflow